# 01 - Curated Transactions

Normaliza `raw.fact_transacciones` y crea `curated_fact_transactions`.

In [ ]:
# Fabric notebook: 01_curated_transactions

from pyspark.sql.functions import (
    col,
    regexp_replace,
    to_timestamp,
    trim,
    upper,
    when,
)


def money(column_name):
    cleaned = regexp_replace(regexp_replace(trim(col(column_name)), "€|EUR|\\s", ""), ",", ".")
    return cleaned.cast("double")


fact = spark.table("raw.fact_transacciones")

curated = (
    fact.select(
        col("fact_id"),
        col("source_system"),
        col("source_file"),
        upper(trim(col("canal_origen"))).alias("channel"),
        col("tipo_movimiento"),
        (col("es_devolucion") == "1").alias("is_return"),
        to_timestamp(col("fecha_movimiento")).alias("movement_date"),
        upper(trim(col("customer_id_std"))).alias("customer_id"),
        upper(trim(col("product_id_std"))).alias("product_id"),
        upper(trim(col("store_id_std"))).alias("store_id"),
        col("doc_id_std"),
        col("line_id_std"),
        col("return_id"),
        col("categoria").alias("category"),
        col("subcategoria"),
        money("cantidad_signed").alias("quantity"),
        money("precio_unitario_num").alias("unit_price"),
        money("descuento_pct_num").alias("discount_pct"),
        money("importe_signed_num").alias("amount"),
        upper(trim(col("moneda"))).alias("moneda"),
        col("flag_customer_in_crm"),
        col("flag_product_in_master"),
        col("flag_fecha_parseada"),
        col("flag_importe_parseado"),
    )
    .withColumn("business_movement", when(col("is_return"), "RETURN").otherwise("SALE"))
)

curated.write.mode("overwrite").format("delta").saveAsTable("curated_fact_transactions")

display(spark.table("curated_fact_transactions").limit(20))
